# Person 1 — Logistic Regression & Data Collection Pipeline

Pipeline Responsibility: Data Collection & Inventory Audit  
Model Assignment: Logistic Regression  

This notebook loads the full labelled dataset from `data/raw`, extracts RGB/HSV/LBP/HOG features, trains Logistic Regression, and writes results to `outputs/`.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODEL_FOLDER = "logistic_regression"
HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in (HERE, *HERE.parents)
        if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()
    ),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

MODEL_DIR = ROOT / "parts" / MODEL_FOLDER
OUTPUT_DIR = MODEL_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "parts"))
from _pipeline import CLASSES, FEATURE_VERSION, SEED, prepare_dataset  # noqa: E402

print("Python:", sys.executable)
print("Project root:", ROOT)
print("Model folder:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

data = prepare_dataset(ROOT)
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_te, y_te = data["X_te"], data["y_te"]
manifest = data["manifest"]
print(f"Unique images: {len(manifest)} | train: {len(X_tr)} | test: {len(X_te)}")
print("Class counts:", data["audit"]["class_counts"])
print("Dataset complete listed counts:", not data["audit"]["download_coverage"]["partial_dataset"])
print("Feature version:", FEATURE_VERSION)


In [ ]:
from sklearn.linear_model import LogisticRegression

print("--- Person 1: Logistic Regression ---")
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", random_state=SEED),
)

start_time = time.perf_counter()
model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average="macro", zero_division=0)
cm = confusion_matrix(y_te, preds, labels=CLASSES)

print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)
print(classification_report(y_te, preds, labels=CLASSES, zero_division=0))

clf = model.named_steps["logisticregression"]
coefs = clf.coef_[0]
metrics = {
    "model_name": "Logistic Regression",
    "pipeline_stage": "Data Collection & Inventory",
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "top_healthy_features": np.argsort(coefs)[:10].tolist(),
    "top_unhealthy_features": np.argsort(coefs)[-10:][::-1].tolist(),
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "classes": CLASSES,
    "feature_version": FEATURE_VERSION,
}
(OUTPUT_DIR / "logistic_regression_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
joblib.dump(model, OUTPUT_DIR / "logistic_regression_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)
